# Full Fine-tuning vs. Parameter-Efficient Fine-tuning (PEFT) — Code Companion

This notebook accompanies **Topic: Full Fine-tuning vs Parameter-Efficient Fine-tuning (PEFT)**.

The slides explain *why* PEFT is so much cheaper than full fine-tuning. This notebook
makes that difference concrete with real numbers and a hands-on (tiny, from-scratch)
demonstration of what "freezing" parameters actually means in code — using the **EFCC
question-answer dataset** (`efcc_test_dataset.json`) as our fine-tuning target.

**What you'll do:**
1. Compute exact trainable-parameter counts for full fine-tuning vs. LoRA, using Llama
   3.1 (8B)'s real architecture dimensions.
2. Estimate the memory each approach needs, including optimizer state.
3. Build a tiny neural network from scratch and literally freeze most of its parameters,
   so you can see the mechanism, not just the theory.
4. Compare the real Hugging Face code for full fine-tuning vs. Unsloth's PEFT setup on the EFCC dataset.

Sections 1-3 run anywhere with just NumPy. Section 4 is reference code for an environment
with `transformers` + `unsloth` + a GPU.

## 1. Counting Parameters: Full Fine-tuning vs. LoRA

We'll fine-tune Llama 3.1 (8B) on the **EFCC question-answer dataset** (~20 examples of\nlegal-domain QA). Let's use the model's real published architecture dimensions to compute\nexactly how many parameters each approach would train.

In [1]:
# Llama 3.1 (8B) architecture dimensions (approximate, from the published config)
d_model = 4096
num_layers = 32
num_heads = 32
d_ff = 14336          # feed-forward inner dimension (SwiGLU has 3 matrices, not 2)
vocab_size = 128256

def count_full_finetune_params():
    # Per-layer: 4 attention projections (Q, K, V, O) + 3 feed-forward matrices (SwiGLU)
    attn_params = 4 * d_model * d_model
    ffn_params = 3 * d_model * d_ff
    norm_params = 2 * d_model   # 2 RMSNorm layers per block
    per_layer = attn_params + ffn_params + norm_params
    total = per_layer * num_layers
    embedding_params = vocab_size * d_model    # input embedding (output head is often tied)
    return total + embedding_params

full_params = count_full_finetune_params()
print(f"Full fine-tuning trainable parameters: {full_params:,}  (~{full_params/1e9:.1f}B)")

Full fine-tuning trainable parameters: 8,310,226,944  (~8.3B)


In [1]:
# Llama 3.1 (8B) architecture dimensions — re-stated here so this cell runs standalone
# (also defined in the previous cell for "Run All" flow)
d_model = 4096
num_layers = 32
num_heads = 32
d_ff = 14336          # feed-forward inner dimension (SwiGLU has 3 matrices, not 2)
vocab_size = 128256

# --- Full fine-tuning parameter count ---
def count_full_finetune_params():
    attn_params = 4 * d_model * d_model
    ffn_params = 3 * d_model * d_ff
    norm_params = 2 * d_model   # 2 RMSNorm layers per block
    per_layer = attn_params + ffn_params + norm_params
    total = per_layer * num_layers
    embedding_params = vocab_size * d_model
    return total + embedding_params

full_params = count_full_finetune_params()
print(f"Full fine-tuning trainable parameters: {full_params:,}  (~{full_params/1e9:.1f}B)")

# --- LoRA parameter count ---
def count_lora_params(rank=16, target_modules=4):
    # LoRA adds two small matrices (A: d x r, B: r x d) per targeted weight matrix
    per_matrix = d_model * rank + rank * d_model
    per_layer = target_modules * per_matrix
    total = per_layer * num_layers
    return total

lora_params = count_lora_params(rank=16, target_modules=4)   # Q, K, V, O projections
print(f"
LoRA (rank=16, 4 target modules) trainable parameters: {lora_params:,}")
print(f"That's just {lora_params / full_params:.3%} of the full fine-tuning parameter count")


NameError: name 'd_model' is not defined

In [3]:
# How does this scale with rank, and with how many matrices LoRA targets?
print(f"{'rank':>5} | {'targets=4 (attn only)':>22} | {'targets=7 (attn+ffn)':>22}")
print("-" * 55)
for rank in [4, 8, 16, 32, 64]:
    p4 = count_lora_params(rank, target_modules=4)
    p7 = count_lora_params(rank, target_modules=7)
    print(f"{rank:>5} | {p4:>16,} ({p4/full_params:>5.2%}) | {p7:>16,} ({p7/full_params:>5.2%})")

 rank |  targets=4 (attn only) |   targets=7 (attn+ffn)
-------------------------------------------------------
    4 |        4,194,304 (0.05%) |        7,340,032 (0.09%)
    8 |        8,388,608 (0.10%) |       14,680,064 (0.18%)
   16 |       16,777,216 (0.20%) |       29,360,128 (0.35%)
   32 |       33,554,432 (0.40%) |       58,720,256 (0.71%)
   64 |       67,108,864 (0.81%) |      117,440,512 (1.41%)


Even at the high end (rank 64, targeting every matrix), LoRA is still training well under
5% of Llama 3.1 (8B)'s parameters. This is exactly the "less than 1-3% of weights" figure
from the slides, now grounded in real numbers.

## 2. Estimating Memory: Where It Actually Goes

Trainable parameters aren't the only thing eating memory. Modern optimizers like Adam
keep extra state (roughly 2 extra numbers per trainable parameter) alongside gradients.
Let's estimate total memory for both approaches.

In [4]:
BYTES_PER_PARAM_16BIT = 2      # model weights, loaded in 16-bit
BYTES_PER_PARAM_4BIT = 0.5     # weights, quantized to 4-bit (used for the frozen base in QLoRA)
BYTES_PER_GRAD = 2             # gradients, same precision as training
BYTES_PER_ADAM_STATE = 4 * 2   # Adam keeps 2 extra 32-bit numbers per trainable parameter

def estimate_memory_gb(total_params, trainable_params, base_precision_bytes):
    weights = total_params * base_precision_bytes
    grads = trainable_params * BYTES_PER_GRAD
    optimizer = trainable_params * BYTES_PER_ADAM_STATE
    total_bytes = weights + grads + optimizer
    return total_bytes / 1e9   # -> gigabytes

full_ft_gb = estimate_memory_gb(full_params, full_params, BYTES_PER_PARAM_16BIT)
lora_16bit_gb = estimate_memory_gb(full_params, lora_params, BYTES_PER_PARAM_16BIT)
qlora_4bit_gb = estimate_memory_gb(full_params, lora_params, BYTES_PER_PARAM_4BIT)

print(f"Full fine-tuning (16-bit base):  ~{full_ft_gb:.1f} GB")
print(f"LoRA (16-bit frozen base):       ~{lora_16bit_gb:.1f} GB")
print(f"QLoRA (4-bit frozen base):       ~{qlora_4bit_gb:.1f} GB")

Full fine-tuning (16-bit base):  ~99.7 GB
LoRA (16-bit frozen base):       ~16.8 GB
QLoRA (4-bit frozen base):       ~4.3 GB


This is a simplified estimate (it ignores activations, which also matter, and real-world
overhead), but the *shape* of the result is exactly right and matches the "Illustrative
VRAM Needs" slide: full fine-tuning needs far more memory than LoRA, and QLoRA's
4-bit base shrinks things further still — often enough to fit on a single free-tier GPU.

## 3. Freezing Parameters, From Scratch

Parameter counts are one thing — but what does "freezing" a weight actually *mean* in
code? Let's build a tiny two-layer network from scratch in NumPy and manually implement
both full fine-tuning and a LoRA-style update, so the mechanism is fully visible.

In [5]:
import numpy as np
np.random.seed(0)

# A tiny "pretrained" weight matrix (stand-in for one layer of a real model)
d = 8
W_pretrained = np.random.randn(d, d) * 0.1
print("Frozen pretrained weight matrix W (shape):", W_pretrained.shape)
print("Total parameters in W:", W_pretrained.size)

Frozen pretrained weight matrix W (shape): (8, 8)
Total parameters in W: 64


In [6]:
# --- Full fine-tuning: every entry of W is directly trainable ---
def full_finetune_step(W, x, target, lr=0.01):
    y = x @ W
    grad_output = 2 * (y - target)               # d(loss)/d(y) for MSE loss
    grad_W = np.outer(x, grad_output)             # d(loss)/d(W) -- full d x d gradient
    W_updated = W - lr * grad_W
    return W_updated, grad_W.size                 # how many numbers we just updated

x = np.random.randn(d)
target = np.random.randn(d)

W_after_full, n_updated_full = full_finetune_step(W_pretrained.copy(), x, target)
print(f"Full fine-tuning updated all {n_updated_full} entries of W directly.")

Full fine-tuning updated all 64 entries of W directly.


In [7]:
# --- LoRA: W stays frozen; we only train two small matrices A and B ---
r = 2   # LoRA rank, deliberately tiny for this demo
A = np.zeros((d, r))          # LoRA is typically initialized so the update starts at zero...
B = np.random.randn(r, d) * 0.01   # ...one of the two matrices starts random, the other zero

def lora_step(W_frozen, A, B, x, target, lr=0.01):
    # forward pass: frozen path + LoRA path
    y = x @ W_frozen + x @ (A @ B)
    grad_output = 2 * (y - target)

    # backprop ONLY into A and B -- W_frozen never receives a gradient update
    grad_B = np.outer((x @ A), grad_output)
    grad_A = np.outer(x, grad_output @ B.T)

    A_updated = A - lr * grad_A
    B_updated = B - lr * grad_B
    n_updated = A.size + B.size
    return A_updated, B_updated, n_updated

A_after, B_after, n_updated_lora = lora_step(W_pretrained, A, B, x, target)
print(f"LoRA updated only {n_updated_lora} numbers (A: {A.size}, B: {B.size}).")
print(f"W_frozen itself was never touched -- same object, same values, before and after.")
print(f"\nLoRA trained {n_updated_lora / n_updated_full:.1%} as many numbers as full fine-tuning,")
print(f"for this tiny {d}x{d} example.")

LoRA updated only 32 numbers (A: 16, B: 16).
W_frozen itself was never touched -- same object, same values, before and after.

LoRA trained 50.0% as many numbers as full fine-tuning,
for this tiny 8x8 example.


This is the entire idea from the "LoRA Injected Into a Weight Matrix" slide, written out
in full: `full_finetune_step` updates every entry of `W` directly; `lora_step` never
touches `W` at all, only the small `A` and `B` matrices layered on top of it. Scale `d`
up to Llama 3.1 (8B)'s real 4096 and this tiny percentage becomes the huge memory savings
computed in Sections 1 and 2.

## 4. The Real Code, Side by Side — Fine-tuning on the EFCC Dataset

Here's how full fine-tuning vs. Unsloth PEFT looks in practice for our **EFCC dataset**.
Both approaches use the same formatted Alpaca-style data from `efcc_test_dataset.json`;
only the training strategy differs.

> **Note:** both cells need a GPU and `pip install transformers` / `pip install unsloth`
> respectively — they're reference code to run in Colab or your own machine, not meant to
> execute in this notebook.


In [ ]:
# Full fine-tuning on the EFCC dataset: every parameter trains by default
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}  ({trainable/total:.1%})")
# -> prints 100.0%: nothing is frozen by default
# Fine-tune on the EFCC formatted dataset (from Data_Preparation notebook)
# trainer = Trainer(model=model, train_dataset=efcc_dataset, ...)


In [ ]:
# PEFT with Unsloth on the EFCC dataset: base frozen, only LoRA adapters trainable
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing = "unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}  ({trainable/total:.2%})")
# -> prints ~0.2%: the base model is frozen, only LoRA adapters train
# Load the EFCC formatted dataset and fine-tune:
# trainer = SFTTrainer(model=model, train_dataset=efcc_dataset, ...)


## Recap & Try It Yourself

You just:
- Computed real trainable-parameter counts for full fine-tuning vs. LoRA on Llama 3.1 (8B)'s
  actual architecture — the model we'll use to fine-tune on the EFCC dataset.
- Estimated the memory footprint of full fine-tuning, LoRA, and QLoRA, and saw why QLoRA
  fits on a free-tier GPU (important for training on our ~20 EFCC examples).
- Built full fine-tuning and LoRA updates from scratch in NumPy, and watched LoRA leave
  the pretrained weight matrix completely untouched.
- Compared the real `transformers` vs. `unsloth` code for each approach on the EFCC dataset.

**Things to try:**
1. In Section 1, change `target_modules` to 7 (adding the feed-forward matrices) and see
   how much the parameter count grows.
2. In Section 3, increase `r` from 2 to 4 and re-run — confirm the number of updated
   parameters doubles as expected.
3. In Section 2, add a rough activation-memory term (proportional to sequence length ×
   batch size × d_model × num_layers) to make the estimate more complete.
4. Load the formatted EFCC dataset from the Data Preparation notebook and try a real
   QLoRA fine-tuning run with Unsloth.
